# Module 8 • Large Language Models

# Lesson 43 • Large Language Model Foundations, Scaling, and Instruction Tuning

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate to Advanced  
**Execution target:** CPU by default

## Scope
This lesson introduces LLM foundations: causal pretraining, scaling, context windows, instruction data, supervised fine-tuning, preference/alignment stages, evaluation, and multilingual considerations.

The executable core uses a small decoder-only Transformer as a **pedagogical analogue**, not a production LLM.

## Learning Objectives
- Explain causal language-model pretraining and perplexity.
- Distinguish parameters, data scale, compute, and context length.
- Explain scaling-law intuition and its limits.
- Distinguish pretraining from instruction tuning.
- Serialize instruction-response examples and mask prompt tokens during SFT.
- Compare pretrained and instruction-tuned generations.
- Explain context-window limits, catastrophic forgetting, hallucination, retrieval, and safety evaluation.
- Identify Arabic and multilingual LLM considerations.

## Table of Contents
1. What Is an LLM?  
2. Decoder-Only Transformers  
3. Autoregressive Pretraining  
4. Perplexity  
5. Scaling Axes  
6. Scaling-Law Intuition  
7. Context Windows  
8. Data Quality and Tokenization  
9. Instruction Tuning  
10. Preference and Alignment Stages  
11. Offline Corpus  
12. Vocabulary and Dataset  
13. Decoder-Only Transformer  
14. Pretraining  
15. Generation  
16. Instruction Dataset  
17. Response-Only SFT  
18. Before/After Comparison  
19. Context-Length Experiment  
20. Scaling Experiment  
21. Evaluation Beyond Perplexity  
22. Hallucination and Prompt Sensitivity  
23. Retrieval and Tools  
24. Arabic and Multilingual Considerations  
25. Reproducibility  
26. Knowledge Check  
27. Exercises  
28. Summary and Next Lesson

# 1. What Is an LLM?
An LLM is a high-capacity neural language model trained on large corpora, typically with Transformer architectures and self-supervised objectives. "Large" reflects several interacting dimensions rather than parameter count alone.

In [ ]:
import math, random, re, copy, platform
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import Dataset, DataLoader

pd.DataFrame([
    ("Parameters","model capacity"),("Training tokens","data scale"),
    ("Compute","optimization budget"),("Context length","visible history"),
    ("Post-training","instruction/alignment behavior")],
    columns=["Dimension","Role"])

# 2. Decoder-Only Transformers
Decoder-only LMs use causal self-attention: position *t* can attend to earlier tokens but not future ones.

# 3. Autoregressive Pretraining
The model factorizes sequence probability as a product of next-token probabilities. Training minimizes cross-entropy over the correct next token at each position.

In [ ]:
toy_logits=torch.tensor([[2.0,0.5,-0.5]])
toy_target=torch.tensor([0])
float(nn.CrossEntropyLoss()(toy_logits,toy_target))

# 4. Perplexity
Perplexity is `exp(cross_entropy)`. Lower values indicate better predictive fit on the evaluated token distribution, but not necessarily better factuality, safety, reasoning, or instruction following.

In [ ]:
losses=np.array([1.0,2.0,3.0,4.0])
pd.DataFrame({"cross_entropy":losses,"perplexity":np.exp(losses)})

# 5. Scaling Axes
Scaling can increase parameters, training tokens, compute, or context length. Good performance depends on balancing these resources.

In [ ]:
pd.DataFrame([
    ("Parameters","capacity","undertraining risk"),
    ("Tokens","coverage","quality/duplication"),
    ("Compute","optimization","cost"),
    ("Context","long-range conditioning","memory/latency")],
    columns=["Axis","Benefit","Constraint"])

# 6. Scaling-Law Intuition
Empirical scaling laws often show smooth loss improvements over suitable ranges of model/data/compute scale. They are empirical relationships, not guarantees that every downstream capability improves monotonically.

In [ ]:
x=np.logspace(1,4,30); y=5.0*x**(-0.12)+1.2
plt.figure(figsize=(8,5)); plt.plot(x,y); plt.xscale("log")
plt.xlabel("Relative training compute"); plt.ylabel("Synthetic validation loss")
plt.title("Illustrative Scaling-Law Shape"); plt.tight_layout(); plt.show()

# 7. Context Windows
The context window bounds how many tokens the model can condition on simultaneously. Longer context helps with documents, dialogue, retrieval, and code, but dense self-attention incurs quadratic attention-matrix growth.

In [ ]:
lengths=[8,16,32,64,128]
pd.DataFrame({"context_length":lengths,"attention_cells":[n*n for n in lengths]})

# 8. Data Quality and Tokenization
At scale, language identification, quality filtering, deduplication, privacy filtering, contamination analysis, and tokenizer efficiency become central engineering choices. Tokenization affects multilingual fairness, morphology, and effective context length.

# 9. Instruction Tuning
Pretraining teaches sequence prediction. Instruction tuning teaches mappings from human-readable requests to desired responses.

In [ ]:
pd.DataFrame([
    ("Pretraining","raw sequences","next-token prediction"),
    ("SFT","instruction-response pairs","desired response tokens"),
    ("Preference optimization","ranked/chosen responses","preference signal"),
    ("Evaluation","held-out tasks","capability and risk metrics")],
    columns=["Stage","Data","Signal"])

# 10. Preference and Alignment Stages
Post-training may use preference data, direct preference optimization, reinforcement-learning-based methods, safety-specific tuning, and uncertainty calibration. These stages optimize behavior beyond raw next-token likelihood.

# 11. Offline Corpus
The local corpus is intentionally tiny and structured. It demonstrates mechanics, not LLM-scale capability.

In [ ]:
pretraining_sentences=[
"language models predict the next token","transformers use self attention for context",
"large models require data compute and evaluation","instruction tuning teaches models to follow requests",
"context windows limit the visible token history","tokenization changes how text becomes model input",
"evaluation should measure more than perplexity","training data quality affects model behavior",
"retrieval can provide external evidence to a model","multilingual models need balanced language coverage",
"arabic morphology creates rich word forms","tashkeel can preserve important linguistic distinctions",
"model scaling increases capacity but also cost","post training changes how models respond to users",
"safety evaluation should test difficult edge cases","prompt wording can change generated responses"]*10
len(pretraining_sentences)

# 12. Vocabulary and Dataset

In [ ]:
TOKEN_PATTERN=re.compile(r"\b\w+(?:[-']\w+)*\b",flags=re.UNICODE)
def tokenize(text): return TOKEN_PATTERN.findall(text.lower())
SPECIAL=["<PAD>","<UNK>","<BOS>","<EOS>","<INSTR>","<RESP>"]
counts=Counter(t for s in pretraining_sentences for t in tokenize(s))
vocab=SPECIAL+sorted(counts)
token2id={t:i for i,t in enumerate(vocab)}; id2token={i:t for t,i in token2id.items()}
PAD_ID,UNK_ID,BOS_ID,EOS_ID,INSTR_ID,RESP_ID=[token2id[t] for t in SPECIAL]
print("Vocabulary size:",len(vocab))

In [ ]:
def encode_text(text): return [BOS_ID]+[token2id.get(t,UNK_ID) for t in tokenize(text)]+[EOS_ID]
class LMDataset(Dataset):
    def __init__(self,texts): self.data=[torch.tensor(encode_text(t),dtype=torch.long) for t in texts]
    def __len__(self): return len(self.data)
    def __getitem__(self,i): return self.data[i]
def collate_lm(batch):
    m=max(len(x) for x in batch)
    inp=torch.full((len(batch),m-1),PAD_ID,dtype=torch.long)
    lab=torch.full((len(batch),m-1),-100,dtype=torch.long)
    for r,x in enumerate(batch): inp[r,:len(x)-1]=x[:-1]; lab[r,:len(x)-1]=x[1:]
    return {"input_ids":inp,"labels":lab,"padding_mask":inp==PAD_ID}
split=int(.8*len(pretraining_sentences)); train_texts=pretraining_sentences[:split]; val_texts=pretraining_sentences[split:]
train_loader=DataLoader(LMDataset(train_texts),batch_size=16,shuffle=True,collate_fn=collate_lm,generator=torch.Generator().manual_seed(42))
val_loader=DataLoader(LMDataset(val_texts),batch_size=16,shuffle=False,collate_fn=collate_lm)
next(iter(train_loader))["input_ids"].shape

# 13. Decoder-Only Transformer

In [ ]:
class CausalTransformerLM(nn.Module):
    def __init__(self,vocabulary_size,d_model=48,nhead=4,layers=2,ff=96,max_len=96,dropout=.1):
        super().__init__(); self.maximum_length=max_len
        self.tok=nn.Embedding(vocabulary_size,d_model,padding_idx=PAD_ID)
        self.pos=nn.Embedding(max_len,d_model)
        layer=nn.TransformerEncoderLayer(d_model=d_model,nhead=nhead,dim_feedforward=ff,dropout=dropout,activation="gelu",batch_first=True,norm_first=True)
        self.tr=nn.TransformerEncoder(layer,num_layers=layers); self.out=nn.Linear(d_model,vocabulary_size)
    def forward(self,input_ids,padding_mask):
        if input_ids.size(1)>self.maximum_length: raise ValueError("Input exceeds context window")
        p=torch.arange(input_ids.size(1),device=input_ids.device).unsqueeze(0)
        h=self.tok(input_ids)+self.pos(p)
        mask=torch.triu(torch.ones(input_ids.size(1),input_ids.size(1),dtype=torch.bool,device=input_ids.device),diagonal=1)
        h=self.tr(h,mask=mask,src_key_padding_mask=padding_mask)
        return {"logits":self.out(h),"hidden_states":h}
DEVICE=torch.device("cpu"); torch.manual_seed(42)
model=CausalTransformerLM(len(vocab)).to(DEVICE)
print("Parameters:",sum(p.numel() for p in model.parameters()))

# 14. Pretraining

In [ ]:
loss_fn=nn.CrossEntropyLoss(ignore_index=-100)
def seed_all(s=42): random.seed(s); np.random.seed(s); torch.manual_seed(s)
def eval_loss(model,loader):
    model.eval(); vals=[]
    with torch.no_grad():
        for b in loader:
            o=model(b["input_ids"].to(DEVICE),b["padding_mask"].to(DEVICE)); logits=o["logits"]
            vals.append(float(loss_fn(logits.reshape(-1,logits.size(-1)),b["labels"].to(DEVICE).reshape(-1)).item()))
    return float(np.mean(vals))
def train_lm(model,epochs=25,lr=.003):
    opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=1e-4); hist=[]; best=copy.deepcopy(model.state_dict()); best_loss=float("inf")
    for e in range(epochs):
        model.train(); vals=[]
        for b in train_loader:
            opt.zero_grad(); o=model(b["input_ids"].to(DEVICE),b["padding_mask"].to(DEVICE)); logits=o["logits"]
            loss=loss_fn(logits.reshape(-1,logits.size(-1)),b["labels"].to(DEVICE).reshape(-1)); loss.backward(); clip_grad_norm_(model.parameters(),5.0); opt.step(); vals.append(float(loss.item()))
        vl=eval_loss(model,val_loader); hist.append({"epoch":e,"training_loss":np.mean(vals),"validation_loss":vl,"validation_perplexity":math.exp(min(vl,20))})
        if vl<best_loss: best_loss=vl; best=copy.deepcopy(model.state_dict())
    model.load_state_dict(best); return model,pd.DataFrame(hist)
seed_all(); pretrained_model,history=train_lm(model)
history.tail()

In [ ]:
plt.figure(figsize=(8,5)); plt.plot(history["epoch"],history["training_loss"],label="Training"); plt.plot(history["epoch"],history["validation_loss"],label="Validation")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy"); plt.title("Causal LM Pretraining"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
vl=eval_loss(pretrained_model,val_loader)
pd.Series({"validation_loss":vl,"validation_perplexity":math.exp(vl)})

# 15. Generation

In [ ]:
def decode(ids):
    out=[]
    for i in ids:
        t=id2token[int(i)]
        if t=="<EOS>": break
        if t not in {"<PAD>","<BOS>","<INSTR>","<RESP>"}: out.append(t)
    return " ".join(out)
def greedy_generate(model,prompt,max_new=12):
    ids=[BOS_ID]+[token2id.get(t,UNK_ID) for t in tokenize(prompt)]
    model.eval()
    with torch.no_grad():
        for _ in range(max_new):
            if len(ids)>=model.maximum_length: break
            x=torch.tensor([ids],dtype=torch.long,device=DEVICE); o=model(x,x==PAD_ID)
            nxt=int(o["logits"][0,-1].argmax().item()); ids.append(nxt)
            if nxt==EOS_ID: break
    return decode(ids)
greedy_generate(pretrained_model,"instruction tuning")

# 16. Instruction Dataset

In [ ]:
instruction_records=[
("define perplexity","perplexity is the exponential of language model cross entropy"),
("define instruction tuning","instruction tuning trains a model on instruction response examples"),
("what is a context window","a context window is the maximum token sequence visible to the model"),
("why use retrieval","retrieval provides external evidence that can improve grounded answers"),
("what is tokenization","tokenization converts text into the token units used by a model"),
("why evaluate factuality","factuality evaluation checks whether generated claims are supported"),
("what does scaling mean","scaling can increase model parameters training data and compute"),
("why deduplicate data","deduplication reduces repeated training content and contamination risk")]*4
instruction_frame=pd.DataFrame(instruction_records,columns=["instruction","response"])
instruction_frame.head()

In [ ]:
def serialize_instruction(inst,response=None):
    ids=[BOS_ID,INSTR_ID]+[token2id.get(t,UNK_ID) for t in tokenize(inst)]+[RESP_ID]
    if response is not None: ids += [token2id.get(t,UNK_ID) for t in tokenize(response)]+[EOS_ID]
    return ids
class InstrDataset(Dataset):
    def __init__(self,frame):
        self.data=[]
        for r in frame.itertuples(index=False):
            full=serialize_instruction(r.instruction,r.response); prompt=serialize_instruction(r.instruction,None)
            inp=full[:-1]; lab=full[1:]; start=len(prompt)-1; lab=[x if i>=start else -100 for i,x in enumerate(lab)]
            self.data.append((torch.tensor(inp),torch.tensor(lab)))
    def __len__(self): return len(self.data)
    def __getitem__(self,i): return self.data[i]
def collate_instr(batch):
    m=max(len(x) for x,_ in batch); inp=torch.full((len(batch),m),PAD_ID,dtype=torch.long); lab=torch.full((len(batch),m),-100,dtype=torch.long)
    for r,(x,y) in enumerate(batch): inp[r,:len(x)]=x; lab[r,:len(y)]=y
    return {"input_ids":inp,"labels":lab,"padding_mask":inp==PAD_ID}
instr_loader=DataLoader(InstrDataset(instruction_frame),batch_size=8,shuffle=True,collate_fn=collate_instr,generator=torch.Generator().manual_seed(42))
next(iter(instr_loader))["input_ids"].shape

# 17. Response-Only SFT
Prompt tokens are masked with `-100`; loss focuses on desired response tokens.

In [ ]:
def train_sft(model,epochs=20,lr=.0015):
    opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=1e-4); rows=[]
    for e in range(epochs):
        model.train(); vals=[]
        for b in instr_loader:
            opt.zero_grad(); o=model(b["input_ids"].to(DEVICE),b["padding_mask"].to(DEVICE)); logits=o["logits"]
            loss=loss_fn(logits.reshape(-1,logits.size(-1)),b["labels"].to(DEVICE).reshape(-1)); loss.backward(); clip_grad_norm_(model.parameters(),5.0); opt.step(); vals.append(float(loss.item()))
        rows.append({"epoch":e,"instruction_loss":np.mean(vals)})
    return model,pd.DataFrame(rows)
seed_all(); instruction_model,sft_history=train_sft(copy.deepcopy(pretrained_model).to(DEVICE))
sft_history.tail()

# 18. Before/After Comparison

In [ ]:
def generate_response(model,instruction,max_new=18):
    ids=serialize_instruction(instruction,None); start=len(ids); model.eval()
    with torch.no_grad():
        for _ in range(max_new):
            if len(ids)>=model.maximum_length: break
            x=torch.tensor([ids],dtype=torch.long,device=DEVICE); o=model(x,x==PAD_ID); nxt=int(o["logits"][0,-1].argmax().item()); ids.append(nxt)
            if nxt==EOS_ID: break
    return decode(ids[start:])
rows=[]
for q in ["define perplexity","what is a context window","why use retrieval","what does scaling mean"]:
    rows.append({"instruction":q,"pretrained":generate_response(pretrained_model,q),"instruction_tuned":generate_response(instruction_model,q)})
pd.DataFrame(rows)

# 19. Context-Length Experiment

In [ ]:
lengths=[8,16,32,64,pretrained_model.maximum_length]
pd.DataFrame({"context_length":lengths,"quadratic_attention_cells":[n*n for n in lengths]})

Truncation can discard relevant evidence. A model accepting a long sequence does not guarantee equal quality at every position.

# 20. Scaling Experiment

In [ ]:
configs=[("Tiny",32,1,64),("Small",48,2,96),("Medium",64,3,128)]
rows=[]
for name,d,l,ff in configs:
    seed_all(); m=CausalTransformerLM(len(vocab),d_model=d,nhead=4,layers=l,ff=ff).to(DEVICE)
    opt=torch.optim.Adam(m.parameters(),lr=.003)
    for _ in range(4):
        m.train()
        for b in train_loader:
            opt.zero_grad(); o=m(b["input_ids"].to(DEVICE),b["padding_mask"].to(DEVICE)); logits=o["logits"]
            loss=loss_fn(logits.reshape(-1,logits.size(-1)),b["labels"].to(DEVICE).reshape(-1)); loss.backward(); opt.step()
    rows.append({"model":name,"parameters":sum(p.numel() for p in m.parameters()),"validation_loss":eval_loss(m,val_loader)})
scaling_results=pd.DataFrame(rows); scaling_results

In [ ]:
plt.figure(figsize=(8,5)); plt.plot(scaling_results["parameters"],scaling_results["validation_loss"],marker="o")
plt.xlabel("Parameter count"); plt.ylabel("Validation loss"); plt.title("Tiny-Scale Capacity Experiment"); plt.tight_layout(); plt.show()

Tiny experiments can violate large-scale trends because optimization noise and data size dominate. The purpose is methodological rather than predictive.

# 21. Evaluation Beyond Perplexity

In [ ]:
pd.DataFrame([
("Perplexity","predictive fit","not instruction quality"),
("Task accuracy","task success","task coverage"),
("Human preference","usefulness","cost and subjectivity"),
("Factuality","claim support","requires evidence"),
("Robustness","stability","prompt-space coverage"),
("Safety tests","risk behavior","scenario coverage")],
columns=["Measure","Strength","Limitation"])

# 22. Hallucination and Prompt Sensitivity
Hallucination refers to generated content that is unsupported, fabricated, or inconsistent with available evidence. Model confidence is not factual certainty. Semantically similar prompts can also yield different outputs.

In [ ]:
variants=["define perplexity","what does perplexity mean","explain perplexity briefly"]
pd.DataFrame({"prompt":variants,"response":[generate_response(instruction_model,p) for p in variants]})

# 23. Retrieval and Tools
LLM systems often combine generation with search, retrieval, calculators, code execution, databases, or APIs. Tools can reduce some limitations but introduce orchestration, security, and grounding requirements.

# 24. Arabic and Multilingual Considerations
Multilingual performance depends on pretraining data coverage, tokenizer efficiency, morphology, dialect distribution, instruction-data quality, and evaluation breadth.

In [ ]:
pd.DataFrame([
("وَسَيَكْتُبُونَهَا","fully vocalized complex word","morphology and clitics"),
("بِالْمَدْرَسَةِ","preposition + article + noun","segmentation efficiency"),
("كِتَابُهُمَا","noun + dual pronoun","agreement and morphology")],
columns=["Form","Description","LLM issue"])

For fully vocalized Arabic tasks, tashkeel should be preserved consistently in pretraining examples, instruction data, evaluation, and inference whenever it is part of the intended task. Removing it changes the input distribution and can erase meaningful distinctions.

# 25. Reproducibility

In [ ]:
pd.Series({
"module":"Module 8 • Large Language Models",
"lesson":"Lesson 43",
"offline model":"decoder-only Transformer",
"parameters":sum(p.numel() for p in pretrained_model.parameters()),
"context window":pretrained_model.maximum_length,
"pretraining examples":len(train_texts),
"instruction examples":len(instruction_frame),
"device":str(DEVICE),"seed":42,"python":platform.python_version(),"torch":torch.__version__},name="Lesson 43 experiment")

# 26. Knowledge Check
1. What makes a language model "large"?  
2. What is the causal LM objective?  
3. Why is perplexity insufficient for complete LLM evaluation?  
4. What dimensions can be scaled?  
5. What is a context window?  
6. Why is long-context attention expensive?  
7. What is instruction tuning?  
8. Why mask prompt tokens in response-focused SFT?  
9. How does SFT differ from preference optimization?  
10. What is catastrophic forgetting?  
11. What is hallucination?  
12. Why can prompt wording affect output?  
13. How can retrieval complement an LLM?  
14. Why does tokenizer efficiency matter for multilingual models?  
15. Why does tashkeel policy matter for Arabic LLM tasks?

# 27. Exercises
## Exercise 1 — Model Scaling
Compare three model sizes using identical training tokens.

## Exercise 2 — Data Scaling
Hold model size fixed and vary pretraining data.

## Exercise 3 — Context Length
Measure runtime as context length increases.

## Exercise 4 — Instruction Masking
Compare response-only loss with loss over the whole example.

## Exercise 5 — Prompt Variants
Measure stability across semantically equivalent prompts.

## Exercise 6 — Catastrophic Forgetting
Evaluate pretraining-domain perplexity before and after SFT.

## Exercise 7 — PEFT
Repeat SFT with LoRA.

## Exercise 8 — Retrieval
Add retrieved context before the instruction.

## Exercise 9 — Arabic Instructions
Create fully vocalized Arabic instruction-response examples.

## Exercise 10 — Evaluation Card
Document capability, factuality, robustness, and safety metrics.

## Challenge Exercises
1. Implement top-k and top-p decoding.  
2. Compare causal LM SFT with encoder–decoder instruction tuning.  
3. Add pairwise preference data and a simple preference loss.  
4. Evaluate multilingual instruction following across English and Arabic.  
5. Build a small retrieval-augmented instruction-following system.

# 28. Summary and Next Lesson
In this lesson:
- LLMs were characterized through parameters, data, compute, and context.
- Causal pretraining and perplexity were implemented.
- Scaling-law intuition and its limitations were discussed.
- Data quality and tokenizer efficiency were connected to model scale.
- Instruction tuning was separated from pretraining.
- Response-only supervised fine-tuning was implemented.
- Before/after generations were compared.
- Context-window and tiny scaling experiments were demonstrated.
- Hallucination, prompt sensitivity, retrieval, safety, multilinguality, and Arabic tashkeel were integrated.

## Next Lesson
**Lesson 44: Prompt Engineering, In-Context Learning, and Structured Outputs** covers zero-shot and few-shot prompting, demonstrations, roles, delimiters, structured schemas, prompt robustness, and systematic prompt evaluation.

# References
- Vaswani, A. et al. *Attention Is All You Need*.
- Brown, T. et al. *Language Models are Few-Shot Learners*.
- Kaplan, J. et al. *Scaling Laws for Neural Language Models*.
- Hoffmann, J. et al. *Training Compute-Optimal Large Language Models*.
- Ouyang, L. et al. *Training Language Models to Follow Instructions with Human Feedback*.
- Wei, J. et al. *Finetuned Language Models Are Zero-Shot Learners*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.